# Support Vector Machine (SVM)

**IT25103271 - Kobewaththa K.A.Y.R**

Dataset: `processed_wine_quality.csv`

In [21]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve)
from sklearn.decomposition import PCA
import joblib

import warnings
warnings.filterwarnings('ignore')

# Step 1: Load Dataset
data_path = 'processed_wine_quality.csv'
df = pd.read_csv(data_path)
feature_cols = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 
                'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 
                'pH', 'sulphates', 'alcohol', 'TotalAcidity']

X = df[feature_cols]
y = (df['quality'] >= 6).astype(int)

In [22]:
# Step 2: Preprocess and Scale Data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Step 3: Hyperparameter Tuning with GridSearchCV
print("\n[INFO] Starting GridSearchCV for Support Vector Machine (SVM)...")
param_grid = {'C': [0.1, 1, 10], 'kernel': ['rbf'], 'gamma': ['scale', 0.1]}

base_svm = SVC(probability=True, random_state=42)
grid_search = GridSearchCV(estimator=base_svm, param_grid=param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("\n[RESULTS] Best Parameters:", grid_search.best_params_)
print(f"[RESULTS] Total Support Vectors: {len(best_model.support_vectors_)}")



[INFO] Starting GridSearchCV for Support Vector Machine (SVM)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


In [ ]:
# Step 4: Stratified 5-Fold Cross-Validation Evaluation
print("\n" + "="*50)
print(" STRATIFIED 5-FOLD CROSS-VALIDATION PERFORMANCE ")
print("="*50)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_acc = cross_val_score(best_model, X_scaled, y, cv=skf, scoring='accuracy')
cv_f1 = cross_val_score(best_model, X_scaled, y, cv=skf, scoring='f1')
cv_auc = cross_val_score(best_model, X_scaled, y, cv=skf, scoring='roc_auc')

print(f" 5-Fold Mean Accuracy: {cv_acc.mean():.4f} (+/- {cv_acc.std():.4f})")
print(f" 5-Fold Mean F1 Score: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")
print(f" 5-Fold Mean ROC AUC : {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})")


In [ ]:
# Step 5: Evaluate Model Performance on Test Set
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("\n" + "="*40)
print(" SUPPORT VECTOR MACHINE TEST EVALUATION ")
print("="*40)
print(f" Accuracy : {acc:.4f}")
print(f" Precision: {prec:.4f}")
print(f" Recall   : {rec:.4f}")
print(f" F1 Score : {f1:.4f}")
print(f" ROC AUC  : {auc:.4f}")


In [ ]:
# Step 6: Visualizations
# 6.1 Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low/Med (0)', 'High (1)'])
disp.plot(cmap='Blues', ax=axes[0])
axes[0].set_title("Confusion Matrix - Support Vector Machine")

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='purple', lw=2, label=f'ROC Curve (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve - Support Vector Machine')
axes[1].legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Save Model
os.makedirs('saved_models', exist_ok=True)
joblib.dump(best_model, 'saved_models/svm_best.pkl')
print("\n[INFO] Best SVM model saved to saved_models/svm_best.pkl")